# ThaiEduFrame — All-in-One Colab Notebook
# เทรน + ประเมินผล LLM ภาษาไทย ในไฟล์เดียว (สำหรับ Google Colab)

รันจากบนลงล่างได้เลย: **ติดตั้ง → เตรียมข้อมูล → เทรน (LoRA/QLoRA) → ประเมินผล (EM/F1)**

### วิธีเริ่ม / How to start
1. เมนู **Runtime → Change runtime type → GPU (T4)** แล้วกด Save
2. รันเซลล์แรกเพื่อติดตั้งไลบรารี
3. ตอนถึงเซลล์ "เตรียมข้อมูล" ให้อัปโหลด `thai_edu_qa.jsonl`
4. รันเซลล์ที่เหลือตามลำดับ

> **⚠ หมายเหตุสำคัญ**
> - GPU T4 ฟรีมีแรม 16 GB — เทรนโมเดล 7B แบบปกติจะ **out-of-memory** จึงตั้ง `USE_4BIT = True` (QLoRA) เป็นค่าเริ่มต้นให้พอดีกับ T4
> - ชุดข้อมูลตัวอย่าง 30 แถวใช้ **ทดสอบว่า pipeline เดินครบเท่านั้น** ผลที่ได้จะ overfit ไม่มีความหมายเชิงวิจัย — ผลจริงต้องใช้ชุด ~1,500 แถว

---

## 0. ติดตั้งไลบรารี + ตรวจ GPU / Install & GPU check

In [ ]:
!pip install -q -U "transformers>=4.40" "peft>=0.10" "accelerate>=0.30" "bitsandbytes>=0.43" datasets pythainlp pandas

In [ ]:
import torch
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {name}  |  VRAM: {vram:.1f} GB")
else:
    print("\u26a0  ไม่พบ GPU — ไปที่ Runtime > Change runtime type > GPU ก่อน")

(ออปชัน) ถ้าโหลดโมเดลแล้วเจอ error เรื่องสิทธิ์ ให้ล็อกอิน Hugging Face ก่อน — ส่วนใหญ่ Typhoon-7B โหลดได้โดยไม่ต้องล็อกอิน

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()

---
## 1. ตั้งค่าทั้งหมด / Configuration

ปุ่มหมุนทั้งหมดอยู่ตรงนี้ที่เดียว แก้แล้วรันเซลล์นี้ใหม่

In [ ]:
# ----- Model -----
BASE_MODEL  = "scb10x/typhoon-7b"          # เปลี่ยนเป็น "bigscience/bloom-7b1" เพื่อเทียบ baseline
USE_4BIT    = True                         # QLoRA — True = พอดีกับ T4 16 GB

# ----- Data -----
DATA_FILE   = "thai_edu_qa.jsonl"

# ----- Output -----
OUTPUT_DIR  = "./checkpoints/typhoon-thai-edu"
ADAPTER_DIR = OUTPUT_DIR + "/final"

# ----- Run switches -----
RUN_TRAINING = True                        # False = ข้ามการเทรน ใช้ adapter ที่มีอยู่แล้ว

# ----- LoRA hyperparameters (ตามเปเปอร์อ้างอิง) -----
LORA_R, LORA_ALPHA, LORA_DROPOUT = 128, 256, 0.1

# ----- Training -----
NUM_EPOCHS  = 10
BATCH_SIZE  = 1                            # T4 -> เล็กไว้ก่อน
GRAD_ACCUM  = 8                            # batch ที่แท้จริง = BATCH_SIZE * GRAD_ACCUM
MAX_LENGTH  = 1024
LR          = 2e-4

# ----- Inference / Eval -----
SPLIT          = "test"
MAX_NEW_TOKENS = 256
TEMPERATURE    = 0.0                       # 0.0 = greedy

# T4 (Turing) ไม่รองรับ bf16 -> ใช้ fp16 อัตโนมัติ
BF16_OK = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
COMPUTE_DTYPE = torch.bfloat16 if BF16_OK else torch.float16
print("compute dtype:", "bfloat16" if BF16_OK else "float16", "| 4-bit:", USE_4BIT)

from pathlib import Path
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

---
## 2. เตรียมข้อมูล / Get the data

หาไฟล์อัตโนมัติ ถ้าไม่เจอจะเปิดหน้าต่างให้อัปโหลด

In [ ]:
import os
candidates = [DATA_FILE, f"/content/{DATA_FILE}", f"data/{DATA_FILE}", f"../data/{DATA_FILE}"]
DATA_PATH = next((c for c in candidates if os.path.exists(c)), None)

if DATA_PATH is None:
    try:
        from google.colab import files
        print(f"\u2b06  กรุณาอัปโหลด {DATA_FILE} ...")
        up = files.upload()
        DATA_PATH = list(up.keys())[0]
    except Exception as e:
        raise FileNotFoundError(
            f"ไม่พบ {DATA_FILE} — อัปโหลดไฟล์เข้าแถบ Files ทางซ้ายก่อน") from e

print("Using data:", DATA_PATH)

In [ ]:
import pandas as pd
df = pd.read_json(DATA_PATH, lines=True)
print("total rows:", len(df))
print(df["split"].value_counts().to_dict())
df.head(3)

---
## 3. ข้อความ + เมตริก / Preprocessing & metrics

`clean_thai` และ F1 (PyThaiNLP **newmm**) เหมือนกับตอนเทรน — ความสม่ำเสมอของ tokenizer คือข้อกำหนดเฉพาะของภาษาไทย

In [ ]:
import re, unicodedata, json
from collections import Counter
from pythainlp.tokenize import word_tokenize


def tokenize(text: str):
    return [t for t in word_tokenize(text, engine="newmm") if t.strip()]


def clean_thai(text: str) -> str:
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"[\u200b\u200c\u200d\ufeff]", "", text)
    text = re.sub(r"<[^>]+>", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def normalize_answer(s: str) -> str:
    s = clean_thai(s)
    s = re.sub(r"[\.\,\!\?\:\;\(\)\[\]\"\\\'\u2018\u2019\u201c\u201d]", "", s)
    return s.strip().lower()


def exact_match(pred: str, gold: str) -> float:
    return float(normalize_answer(pred) == normalize_answer(gold))


def f1_score_tokens(pred: str, gold: str) -> float:
    pt, gt = tokenize(normalize_answer(pred)), tokenize(normalize_answer(gold))
    if not pt or not gt:
        return float(pt == gt)
    common = Counter(pt) & Counter(gt)
    n = sum(common.values())
    if n == 0:
        return 0.0
    p, r = n / len(pt), n / len(gt)
    return 2 * p * r / (p + r)


def evaluate_predictions(gold, pred):
    em = [exact_match(p, g)     for p, g in zip(pred, gold)]
    f1 = [f1_score_tokens(p, g) for p, g in zip(pred, gold)]
    return {"EM": sum(em)/len(em)*100, "F1": sum(f1)/len(f1)*100, "n": len(em)}


# sanity tests — อย่าเชื่อเมตริกถ้ายังไม่ผ่าน
assert exact_match("สวัสดี", "สวัสดี") == 1.0
assert exact_match("สวัสดี", "สวัสดีครับ") == 0.0
assert f1_score_tokens("สวัสดี", "สวัสดี") == 1.0
assert 0.0 < f1_score_tokens("GPA ต้องไม่ต่ำกว่า 2.00", "นิสิตต้องมี GPA ไม่ต่ำกว่า 2.00") < 1.0
print("\u2713 metric sanity tests passed")

### Prompt template + dataset / สร้าง prompt และชุดข้อมูล

เทรนใช้ template ที่ **มีคำตอบ** ส่วน inference ใช้ template ที่ **เว้นคำตอบว่าง**

In [ ]:
from datasets import Dataset

TRAIN_TEMPLATE = """### คำสั่ง (Instruction):
ตอบคำถามต่อไปนี้โดยอ้างอิงจากบริบทที่ให้

### บริบท (Context):
{context}

### คำถาม (Question):
{question}

### คำตอบ (Answer):
{answer}"""

INFER_TEMPLATE = """### คำสั่ง (Instruction):
ตอบคำถามต่อไปนี้โดยอ้างอิงจากบริบทที่ให้

### บริบท (Context):
{context}

### คำถาม (Question):
{question}

### คำตอบ (Answer):
"""

def make_text(ex):
    return {"text": TRAIN_TEMPLATE.format(
        context=clean_thai(ex["context"]),
        question=clean_thai(ex["question"]),
        answer=clean_thai(ex["answer"]),
    )}

train_rows = df[df["split"] == "train"].to_dict("records")
val_rows   = df[df["split"] == "val"].to_dict("records")
train_ds = Dataset.from_list(train_rows).map(make_text)
val_ds   = Dataset.from_list(val_rows).map(make_text)
print(f"train: {len(train_ds)}  | val: {len(val_ds)}")
print("\n--- ตัวอย่าง prompt (train) ---\n")
print(train_ds[0]["text"])

---
## 4. โหลดโมเดล + ใส่ LoRA / Load model & apply LoRA

ถ้า `USE_4BIT=True` จะโหลดแบบ 4-bit (QLoRA) เพื่อให้พอดีกับ T4

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if USE_4BIT:
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=COMPUTE_DTYPE,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, quantization_config=bnb_cfg, device_map="auto")
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
else:
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, torch_dtype=COMPUTE_DTYPE, device_map="auto")
    model.gradient_checkpointing_enable()

lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], bias="none",
)
model = get_peft_model(model, lora_cfg)
model.config.use_cache = False          # ต้องปิดตอนเทรน (ใช้คู่กับ gradient checkpointing)
model.print_trainable_parameters()

---
## 5. เทรน / Train

batch จริง = `BATCH_SIZE * GRAD_ACCUM` ใช้ `paged_adamw_8bit` ตอน QLoRA เพื่อประหยัดแรม

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

def tokenize_fn(batch):
    out = tokenizer(batch["text"], max_length=MAX_LENGTH, truncation=True, padding=False)
    out["labels"] = out["input_ids"].copy()
    return out

train_tok = train_ds.map(tokenize_fn, batched=True, remove_columns=train_ds.column_names)
val_tok   = val_ds.map(tokenize_fn,   batched=True, remove_columns=val_ds.column_names)

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR, warmup_ratio=0.05, weight_decay=0.01,
    logging_steps=5,
    eval_strategy="epoch", save_strategy="no",
    fp16=(COMPUTE_DTYPE == torch.float16),
    bf16=(COMPUTE_DTYPE == torch.bfloat16),
    optim="paged_adamw_8bit" if USE_4BIT else "adamw_torch",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none",
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_tok, eval_dataset=val_tok,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

if RUN_TRAINING:
    trainer.train()
else:
    print("RUN_TRAINING=False -> ข้ามการเทรน")

### บันทึก adapter / Save the LoRA adapter

In [ ]:
if RUN_TRAINING:
    final = Path(OUTPUT_DIR) / "final"
    model.save_pretrained(final)
    tokenizer.save_pretrained(final)
    print("saved adapter ->", final)

### คืนหน่วยความจำก่อนประเมินผล / Free VRAM before evaluation

เคลียร์โมเดลตอนเทรนออกจาก GPU แล้วโหลดใหม่สำหรับ inference (กัน OOM บน T4)

In [ ]:
import gc
del trainer, model
gc.collect()
torch.cuda.empty_cache()
print("freed.")

---
## 6. โหลดโมเดลสำหรับ inference / Load model for evaluation

In [ ]:
from peft import PeftModel

def load_for_inference(base_model, adapter_dir, use_4bit):
    tok = AutoTokenizer.from_pretrained(adapter_dir)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    if use_4bit:
        bnb_cfg = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=COMPUTE_DTYPE, bnb_4bit_use_double_quant=True)
        base = AutoModelForCausalLM.from_pretrained(
            base_model, quantization_config=bnb_cfg, device_map="auto")
    else:
        base = AutoModelForCausalLM.from_pretrained(
            base_model, torch_dtype=COMPUTE_DTYPE, device_map="auto")
    m = PeftModel.from_pretrained(base, adapter_dir)
    m.config.use_cache = True
    m.eval()
    return m, tok

assert Path(ADAPTER_DIR).exists(), f"ไม่พบ adapter ที่ {ADAPTER_DIR} — เทรนก่อน หรือชี้ path ให้ถูก"
model, tokenizer = load_for_inference(BASE_MODEL, ADAPTER_DIR, USE_4BIT)
print("model ready for inference.")

In [ ]:
@torch.inference_mode()
def predict_one(context: str, question: str) -> str:
    prompt = INFER_TEMPLATE.format(context=context, question=question)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                       max_length=MAX_LENGTH).to(model.device)
    gen = dict(max_new_tokens=MAX_NEW_TOKENS,
               do_sample=(TEMPERATURE > 0),
               temperature=TEMPERATURE if TEMPERATURE > 0 else 1.0,
               pad_token_id=tokenizer.pad_token_id)
    out = model.generate(**inputs, **gen)
    new = out[0, inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(new, skip_special_tokens=True).strip()
    for stop in ["###", "\n\n\n"]:
        if stop in text:
            text = text.split(stop)[0].strip()
    return text

# ทดสอบ 1 ข้อ
d = df[df["split"] == SPLIT].iloc[0]
print("Q   :", d["question"])
print("Gold:", d["answer"])
print("Pred:", predict_one(d["context"], d["question"]))

---
## 7. ประเมินผลทั้งชุดทดสอบ / Evaluate the test set

In [ ]:
test_df = df[df["split"] == SPLIT].copy().reset_index(drop=True)

preds = []
for i, row in test_df.iterrows():
    preds.append(predict_one(row["context"], row["question"]))
    if (i + 1) % 10 == 0 or i == len(test_df) - 1:
        print(f"  [{i+1}/{len(test_df)}] done")
test_df["prediction"] = preds

test_df["em"] = [exact_match(p, g)     for p, g in zip(test_df["prediction"], test_df["answer"])]
test_df["f1"] = [f1_score_tokens(p, g) for p, g in zip(test_df["prediction"], test_df["answer"])]

overall = {"split": SPLIT, "n": len(test_df),
           "EM": round(test_df["em"].mean()*100, 2),
           "F1": round(test_df["f1"].mean()*100, 2)}
print("=" * 46)
print(f"FINAL  ({SPLIT}, n={overall['n']})   EM={overall['EM']}   F1={overall['F1']}")
print("=" * 46)

### บันทึกผล / Save predictions & metrics

In [ ]:
out_dir = Path("./eval_results"); out_dir.mkdir(parents=True, exist_ok=True)
test_df[["id", "question", "answer", "prediction", "em", "f1"]].to_json(
    out_dir / f"predictions_{SPLIT}.jsonl", orient="records", lines=True, force_ascii=False)
with open(out_dir / f"metrics_{SPLIT}.json", "w", encoding="utf-8") as f:
    json.dump({**overall, "adapter": ADAPTER_DIR, "base_model": BASE_MODEL,
               "use_4bit": USE_4BIT}, f, indent=2, ensure_ascii=False)
print("saved ->", out_dir)

# (Colab) ดาวน์โหลดไฟล์ผลลัพธ์
# from google.colab import files
# files.download(str(out_dir / f"metrics_{SPLIT}.json"))

### วิเคราะห์ข้อผิดพลาด / Error analysis — 5 ข้อที่ F1 ต่ำสุด

In [ ]:
for _, r in test_df.sort_values("f1").head(5).iterrows():
    print(f"[{r['id']}]  EM={r['em']:.0f}  F1={r['f1']:.2f}")
    print("  Q    :", r["question"])
    print("  Gold :", r["answer"])
    print("  Pred :", r["prediction"])
    print("-" * 70)

---
## 8. (ออปชัน) เทียบ baseline / Baseline comparison

Retrieval (token-Jaccard) — ถ้าโมเดลชนะไม่ได้แสดงว่ายังไม่เข้าใจภาษาจริง

In [ ]:
train_df = df[df["split"] == "train"].copy()
train_df["ctok"] = train_df["context"].apply(lambda x: set(tokenize(clean_thai(x))))

def retrieve_answer(q):
    qt = set(tokenize(clean_thai(q)))
    if not qt:
        return train_df.iloc[0]["answer"]
    best, ans = -1.0, train_df.iloc[0]["answer"]
    for _, row in train_df.iterrows():
        ct = row["ctok"]
        if not ct:
            continue
        s = len(qt & ct) / len(qt | ct)
        if s > best:
            best, ans = s, row["answer"]
    return ans

test_df["pred_retrieval"] = test_df["question"].apply(retrieve_answer)

results = {
    "Retrieval (Jaccard)": evaluate_predictions(test_df["answer"], test_df["pred_retrieval"]),
    f"{BASE_MODEL.split('/')[-1]} + LoRA": evaluate_predictions(test_df["answer"], test_df["prediction"]),
}
import pandas as pd
res_df = pd.DataFrame(results).T.round(2); res_df.index.name = "Method"
print(res_df)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 4))
x = range(len(res_df)); w = 0.35
em, f1 = res_df["EM"].values, res_df["F1"].values
ax.bar([i - w/2 for i in x], em, w, label="EM", color="#065A82")
ax.bar([i + w/2 for i in x], f1, w, label="F1", color="#F4A261")
ax.set_xticks(list(x)); ax.set_xticklabels(res_df.index, fontsize=9)
ax.set_ylim(0, 110); ax.set_ylabel("Score (%)")
ax.set_title("Fine-tuned vs. retrieval baseline"); ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
for i, (a, b) in enumerate(zip(em, f1)):
    ax.text(i - w/2, a + 2, f"{a:.0f}", ha="center", fontsize=9)
    ax.text(i + w/2, b + 2, f"{b:.0f}", ha="center", fontsize=9)
plt.tight_layout(); plt.show()

---
**🇹🇭 ขั้นต่อไป / Next steps**

- ตัวเลขใน `eval_results/metrics_test.json` คือค่าที่นำไปใส่ **Table II** ของเปเปอร์ (แทนค่า projection)
- อยากเทียบ Bloom: ตั้ง `BASE_MODEL = "bigscience/bloom-7b1"` แล้วรันใหม่ทั้งไฟล์
- ได้ผลจริงต้องใช้ชุดข้อมูล ~1,500 แถว — ชุด 30 แถวนี้ใช้ตรวจว่า pipeline เดินครบเท่านั้น
- ถ้า T4 ยัง OOM ตอนเทรน: ลด `MAX_LENGTH` เป็น 512 หรือ `LORA_R` เป็น 64